# Customer Churn Model Training

End-to-end training pipeline that runs in **Vertex AI Workbench**:

1. Load IBM Telco Customer Churn data from **BigQuery**
2. Clean + feature-engineer
3. Train an **XGBoost** classifier in a scikit-learn pipeline
4. Evaluate (accuracy, ROC-AUC, classification report)
5. Compute **SHAP** global feature importance
6. Save `model.joblib` + `features_meta.json` locally
7. Upload artifacts to **Cloud Storage**
8. Register the model in the BigQuery `model_registry` table
9. (Optional) Register + deploy as a **Vertex AI Endpoint**

Before running, replace the placeholders in the first cell.

In [ ]:
# ===== Configuration: edit these =====
PROJECT_ID   = 'your-project-id'          # e.g. 'churn-platform-prod'
REGION       = 'us-central1'
BQ_DATASET   = 'telco_churn'
BQ_TABLE     = 'customers'
GCS_BUCKET   = 'your-project-id-models'   # bucket must exist in REGION
MODEL_VERSION = 'v1'
REGISTER_VERTEX_ENDPOINT = False          # set True to also deploy to Vertex endpoint (~$0.05/hr)

In [ ]:
# ===== Install dependencies =====
!pip install --quiet --upgrade \
    pandas==2.2.2 numpy==1.26.4 scikit-learn==1.5.1 \
    xgboost==2.1.1 shap==0.46.0 joblib==1.4.2 \
    google-cloud-bigquery==3.25.0 google-cloud-storage==2.18.0 \
    google-cloud-aiplatform==1.71.1 db-dtypes==1.3.0 pandas-gbq==0.23.1
print('Done.')

In [ ]:
import json, os, datetime, joblib
import numpy as np, pandas as pd
from google.cloud import bigquery, storage
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, roc_auc_score, classification_report,
    precision_score, recall_score, f1_score, confusion_matrix
)
from xgboost import XGBClassifier
import shap
print('Imports OK')

## 1. Load data from BigQuery

In [ ]:
bq_client = bigquery.Client(project=PROJECT_ID)
sql = f'SELECT * FROM `{PROJECT_ID}.{BQ_DATASET}.{BQ_TABLE}`'
df = bq_client.query(sql).to_dataframe()
print(f'Rows: {len(df)}, Columns: {len(df.columns)}')
df.head()

## 2. Clean + feature engineering

In [ ]:
# TotalCharges has whitespace strings; convert to float
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'].astype(str).str.strip(), errors='coerce')
df['TotalCharges'] = df['TotalCharges'].fillna(df['TotalCharges'].median())

# Target
df['churn'] = (df['Churn'] == 'Yes').astype(int)

FEATURE_COLUMNS = [
    'gender', 'SeniorCitizen', 'Partner', 'Dependents', 'tenure',
    'PhoneService', 'MultipleLines', 'InternetService',
    'OnlineSecurity', 'OnlineBackup', 'DeviceProtection',
    'TechSupport', 'StreamingTV', 'StreamingMovies',
    'Contract', 'PaperlessBilling', 'PaymentMethod',
    'MonthlyCharges', 'TotalCharges',
]
NUMERIC_FEATURES = ['SeniorCitizen', 'tenure', 'MonthlyCharges', 'TotalCharges']
CATEGORICAL_FEATURES = [c for c in FEATURE_COLUMNS if c not in NUMERIC_FEATURES]

X = df[FEATURE_COLUMNS].copy()
y = df['churn']

print('Class balance:')
print(y.value_counts(normalize=True))

## 3. Train / test split + pipeline

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)
print(f'Train: {len(X_train)}, Test: {len(X_test)}')

preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), NUMERIC_FEATURES),
        ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), CATEGORICAL_FEATURES),
    ]
)

scale_pos_weight = float((y_train == 0).sum() / max((y_train == 1).sum(), 1))
print(f'scale_pos_weight = {scale_pos_weight:.3f}')

clf = XGBClassifier(
    n_estimators=300,
    max_depth=5,
    learning_rate=0.08,
    subsample=0.85,
    colsample_bytree=0.85,
    scale_pos_weight=scale_pos_weight,
    eval_metric='logloss',
    random_state=42,
    n_jobs=-1,
)

pipeline = Pipeline([('preprocess', preprocessor), ('model', clf)])

In [ ]:
pipeline.fit(X_train, y_train)
print('Training complete.')

## 4. Evaluate

In [ ]:
y_pred = pipeline.predict(X_test)
y_proba = pipeline.predict_proba(X_test)[:, 1]

metrics = {
    'accuracy':  round(accuracy_score(y_test, y_pred), 4),
    'roc_auc':   round(roc_auc_score(y_test, y_proba), 4),
    'precision': round(precision_score(y_test, y_pred), 4),
    'recall':    round(recall_score(y_test, y_pred), 4),
    'f1':        round(f1_score(y_test, y_pred), 4),
}
print('Metrics:', json.dumps(metrics, indent=2))
print()
print(classification_report(y_test, y_pred, target_names=['No Churn', 'Churn']))
print('Confusion matrix:')
print(confusion_matrix(y_test, y_pred))

## 5. SHAP global feature importance

In [ ]:
# Get the transformed feature space
preprocessor_fitted = pipeline.named_steps['preprocess']
X_test_transformed = preprocessor_fitted.transform(X_test)

# Feature names after one-hot
feature_names = (
    NUMERIC_FEATURES +
    list(preprocessor_fitted.named_transformers_['cat'].get_feature_names_out(CATEGORICAL_FEATURES))
)

# Use a sample for SHAP to keep runtime reasonable
sample = X_test_transformed[:500]
explainer = shap.TreeExplainer(pipeline.named_steps['model'])
shap_values = explainer.shap_values(sample)

mean_abs_shap = np.abs(shap_values).mean(axis=0)
importance_df = pd.DataFrame({'feature': feature_names, 'importance': mean_abs_shap})
importance_df = importance_df.sort_values('importance', ascending=False).reset_index(drop=True)
print('Top 15 features by SHAP importance:')
print(importance_df.head(15))

## 6. Save artifacts locally

In [ ]:
os.makedirs('model_artifacts', exist_ok=True)

joblib.dump(pipeline, 'model_artifacts/model.joblib')

# Build the global_importance map keyed by the ORIGINAL feature name so the
# Predictor can attribute factors back to user-facing fields.
global_importance = {}
for _, row in importance_df.head(20).iterrows():
    raw_name = row['feature']
    # Map onehot names like 'Contract_Month-to-month' back to base 'Contract'
    base = raw_name.split('_')[0] if '_' in raw_name and raw_name not in NUMERIC_FEATURES else raw_name
    global_importance[raw_name] = float(row['importance'])

meta = {
    'model_version': MODEL_VERSION,
    'trained_at': datetime.datetime.utcnow().isoformat(),
    'algorithm': 'XGBoostClassifier',
    'metrics': metrics,
    'feature_names': feature_names,
    'global_importance': global_importance,
    'feature_columns': FEATURE_COLUMNS,
    'numeric_features': NUMERIC_FEATURES,
    'categorical_features': CATEGORICAL_FEATURES,
}
with open('model_artifacts/features_meta.json', 'w') as f:
    json.dump(meta, f, indent=2)

print('Wrote model_artifacts/model.joblib and features_meta.json')

## 7. Upload to Cloud Storage

In [ ]:
storage_client = storage.Client(project=PROJECT_ID)
bucket = storage_client.bucket(GCS_BUCKET)

for fname in ['model.joblib', 'features_meta.json']:
    blob = bucket.blob(f'churn/{MODEL_VERSION}/{fname}')
    blob.upload_from_filename(f'model_artifacts/{fname}')
    print(f'Uploaded gs://{GCS_BUCKET}/churn/{MODEL_VERSION}/{fname}')

## 8. Register in BigQuery model_registry

In [ ]:
registry_row = {
    'model_version': MODEL_VERSION,
    'algorithm': 'XGBoostClassifier',
    'training_date': datetime.datetime.utcnow().isoformat(),
    'accuracy':       metrics['accuracy'],
    'roc_auc':        metrics['roc_auc'],
    'precision_score':metrics['precision'],
    'recall_score':   metrics['recall'],
    'f1_score':       metrics['f1'],
    'gcs_uri':        f'gs://{GCS_BUCKET}/churn/{MODEL_VERSION}/model.joblib',
    'is_active':      True,
    'notes':          'XGBoost with scale_pos_weight + SHAP explainability',
}
errors = bq_client.insert_rows_json(
    f'{PROJECT_ID}.{BQ_DATASET}.model_registry', [registry_row]
)
print('BQ insert errors:', errors)

## 9. (Optional) Deploy to Vertex AI Endpoint

Skip this section if you want to keep costs at zero — the Cloud Run app can use the bundled `model.joblib` directly. The endpoint costs ~$0.05/hr while it's deployed.

In [ ]:
if REGISTER_VERTEX_ENDPOINT:
    from google.cloud import aiplatform

    aiplatform.init(project=PROJECT_ID, location=REGION)
    artifact_uri = f'gs://{GCS_BUCKET}/churn/{MODEL_VERSION}/'
    model = aiplatform.Model.upload(
        display_name=f'churn-xgb-{MODEL_VERSION}',
        artifact_uri=artifact_uri,
        serving_container_image_uri='us-docker.pkg.dev/vertex-ai/prediction/sklearn-cpu.1-3:latest',
    )
    endpoint = model.deploy(
        machine_type='n1-standard-2',
        min_replica_count=1,
        max_replica_count=1,
    )
    print('Endpoint resource name:', endpoint.resource_name)
    print('Endpoint ID (use in .env as VERTEX_ENDPOINT_ID):', endpoint.name)
else:
    print('Skipping Vertex endpoint deployment. Bundle model.joblib into the Cloud Run image instead.')

## Done

Download `model_artifacts/model.joblib` and `model_artifacts/features_meta.json` to your local machine and place them in the `model_artifacts/` folder of the project before building the Docker image.